# Процеси в Python

У цьому ноутбуці зібрані приклади про процеси, паралельне виконання, міжпроцесну взаємодію та спільну пам'ять.
Йдемо від простіших сценаріїв до більш прикладних.

## Важлива примітка про запуск у Jupyter

У першій code-cell ми створимо `MP = mp.get_context("fork")`.
Це потрібно, щоб приклади з процесами було простіше запускати прямо в Jupyter на macOS і Linux: дочірній процес успадковує поточний стан і може бачити функції, оголошені в notebook.

На що звернути увагу:

- на macOS і Linux такий підхід у notebook зазвичай працює найзручніше;
- на Windows `fork` недоступний, там зазвичай використовують `spawn`;
- якщо у вас Windows, такі приклади краще запускати не з notebook, а з окремих `.py`-файлів через `if __name__ == "__main__":`;
- для звичайних скриптів під різні ОС важливо пам'ятати, що код з multiprocessing часто потребує окремого файлу, а не лише cells у Jupyter.

In [2]:
from __future__ import annotations

import hashlib
import math
import multiprocessing as mp
import os
import random
import socket
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from multiprocessing.connection import Connection
from multiprocessing.shared_memory import SharedMemory
from pprint import pprint

if sys.platform == 'win32':
    raise RuntimeError(
        'У цьому notebook приклади налаштовані під fork-контекст. '
        "На Windows краще запускати їх як окремі .py-скрипти через if __name__ == '__main__':"
    )

MP = mp.get_context('fork')
STOP = 'STOP'

if hasattr(sys, 'set_int_max_str_digits'):
    sys.set_int_max_str_digits(0)

print(f'CPU cores: {os.cpu_count()}')
print(f'Start method for this notebook: {MP.get_start_method()}')

CPU cores: 10
Start method for this notebook: fork


## CPU-bound задачі

Процеси особливо корисні там, де є важкі обчислення на CPU.
У таких задачах важливо не просто зробити код "одночасним", а реально рознести роботу по кількох ядрах.

### Fibonacci: послідовно і паралельно

Цей приклад показує базову ідею на рекурсивному Fibonacci.
Він не про ефективний алгоритм, а про те, як порівняти звичайне виконання і запуск через пул процесів.

In [3]:
def fib(number: int) -> int:
    if number <= 1:
        return number
    return fib(number - 1) + fib(number - 2)


def sequential_fibonacci(numbers: list[int]) -> list[int]:
    return [fib(number) for number in numbers]


def parallel_fibonacci(numbers: list[int]) -> list[int]:
    processes_count = min(len(numbers), os.cpu_count() or 1)
    with MP.Pool(processes=processes_count) as pool:
        return pool.map(fib, numbers)

In [4]:
test_numbers = [35, 41, 40, 38]

start_time = time.time()
seq_result = sequential_fibonacci(test_numbers)
seq_time = time.time() - start_time

start_time = time.time()
par_result = parallel_fibonacci(test_numbers)
par_time = time.time() - start_time

print(f'Час послідовного виконання: {seq_time:.2f} сек')
print(f'Час паралельного виконання: {par_time:.2f} сек')
print(f'Прискорення: {seq_time / par_time:.2f}x')
print(seq_result == par_result)

Час послідовного виконання: 31.67 сек
Час паралельного виконання: 16.75 сек
Прискорення: 1.89x
True


Тут ключова думка проста: якщо задачі незалежні одна від одної, їх зручно роздати процесам.
Але прискорення не завжди буде великим, бо створення процесів теж має свою ціну.

### `ProcessPoolExecutor`: факторіали

Тут ідея та сама, але вже через інший інтерфейс для пулу процесів.
Кожне число обробляється окремо, тому задачі добре підходять для паралельного запуску.

In [5]:
def compute_factorial(number: int) -> int:
    return math.factorial(number)

In [6]:
numbers = [100000, 200000, 300000, 400000, 500000]

with ProcessPoolExecutor(max_workers=4, mp_context=MP) as executor:
    results = list(executor.map(compute_factorial, numbers))

[len(str(value)) for value in results]

[456574, 973351, 1512852, 2067110, 2632342]

Ми не виводимо самі значення факторіалів, бо вони дуже великі.
Натомість дивимось на довжину результатів, щоб переконатися, що обчислення справді відбулися.

### Багаторазове хешування токенів

Це ще один приклад CPU-bound задачі.
Кожен токен багато разів хешується, тому на великій кількості елементів тут добре видно сенс процесів.

In [7]:
def calculate_token_score(token: str) -> int:
    value = token.encode()

    for _ in range(100_000):
        value = hashlib.sha256(value).digest()

    return value[0]


def run_sequential(tokens: list[str]) -> list[int]:
    return [calculate_token_score(token) for token in tokens]


def run_parallel(tokens: list[str]) -> list[int]:
    with ProcessPoolExecutor(max_workers=4, mp_context=MP) as executor:
        results = list(executor.map(calculate_token_score, tokens))
    return results

In [8]:
tokens = [os.urandom(16).hex() for _ in range(200)]

start = time.perf_counter()
sequential_scores = run_sequential(tokens)
sequential_time = time.perf_counter() - start

start = time.perf_counter()
parallel_scores = run_parallel(tokens)
parallel_time = time.perf_counter() - start

print(f'Sequential time: {sequential_time:.2f}s')
print(f'Parallel time:   {parallel_time:.2f}s')
print(f'Results equal:   {sequential_scores == parallel_scores}')
print(f'First scores:    {parallel_scores[:10]}')

Sequential time: 5.72s
Parallel time:   2.16s
Results equal:   True
First scores:    [99, 209, 42, 168, 183, 212, 147, 128, 130, 252]


У таких задачах важливо перевіряти не лише час, а і збіг результатів.
Паралельне виконання має пришвидшувати обчислення, але не змінювати їхній зміст.

## `Queue`: розподіл задач між процесами

Тепер переходимо до міжпроцесної взаємодії.
Черга зручна тоді, коли один або кілька процесів кладуть задачі, а воркери їх забирають і повертають результати.

In [9]:
STOP = None


def is_dark_image(pixels: list[list[int]]) -> bool:
    total_brightness = 0
    total_pixels = 0

    for row in pixels:
        for pixel in row:
            total_brightness += pixel
            total_pixels += 1

    average_brightness = total_brightness / total_pixels

    return average_brightness < 80


def worker(task_queue: mp.Queue, result_queue: mp.Queue) -> None:
    while True:
        task = task_queue.get()

        if task is STOP:
            break

        image_id, pixels = task
        is_dark = is_dark_image(pixels)

        result_queue.put((image_id, is_dark))


def generate_image(width: int, height: int, max_brightness: int) -> list[list[int]]:
    return [[random.randint(0, max_brightness) for _ in range(width)] for _ in range(height)]

In [10]:
images = [
    (1, generate_image(300, 300, max_brightness=60)),
    (2, generate_image(300, 300, max_brightness=255)),
    (3, generate_image(300, 300, max_brightness=50)),
    (4, generate_image(300, 300, max_brightness=255)),
    (5, generate_image(300, 300, max_brightness=70)),
    (6, generate_image(300, 300, max_brightness=255)),
    (7, generate_image(300, 300, max_brightness=40)),
    (8, generate_image(300, 300, max_brightness=255)),
]

workers_count = 4
task_queue: mp.Queue = MP.Queue()
result_queue: mp.Queue = MP.Queue()

workers = [MP.Process(target=worker, args=(task_queue, result_queue)) for _ in range(workers_count)]

for process in workers:
    process.start()

for image in images:
    task_queue.put(image)

for _ in workers:
    task_queue.put(STOP)

results = [result_queue.get() for _ in images]

for process in workers:
    process.join()

for image_id, is_dark in sorted(results):
    status = 'dark' if is_dark else 'normal'
    print(f'image_{image_id}: {status}')

image_1: dark
image_2: normal
image_3: dark
image_4: normal
image_5: dark
image_6: normal
image_7: dark
image_8: normal


Тут кожне "зображення" потрапляє в чергу задач, а результати повертаються окремо.
Такий шаблон дуже типовий для паралельної обробки великої кількості незалежних елементів.

## `Pipe`: простий канал між двома процесами

`Pipe` добре підходить для прямого обміну даними між двома процесами.
Спочатку подивимось на найпростіший приклад із повідомленнями.

In [11]:
STOP = 'STOP'


def sender(conn: Connection, messages: list[str]):
    for msg in messages:
        print(f'Відправлено: {msg}')
        conn.send(msg)
        time.sleep(1)
    conn.close()


def receiver(conn: Connection, num_messages: int):
    for _ in range(num_messages):
        msg = conn.recv()
        print(f'Отримано: {msg}')
    conn.close()

In [12]:
parent_conn, child_conn = MP.Pipe()
messages = ['Привіт!', 'Як справи?', 'Побачимось!']

sender_process = MP.Process(target=sender, args=(parent_conn, messages))
receiver_process = MP.Process(target=receiver, args=(child_conn, len(messages)))

sender_process.start()
receiver_process.start()

sender_process.join()
receiver_process.join()

Відправлено: Привіт!
Отримано: Привіт!
Відправлено: Як справи?
Отримано: Як справи?
Відправлено: Побачимось!
Отримано: Побачимось!


Тут добре видно базову ідею `Pipe`: один процес надсилає, інший читає.
Це простий варіант, коли не потрібна черга на багато воркерів.

### `Pipe` для запиту й відповіді

Тепер приклад трохи практичніший: один процес надсилає події, інший їх перевіряє і повертає результат назад.

In [13]:
def validate_event(event: dict[str, str]) -> dict[str, str | bool]:
    is_valid = bool(event.get('user_id')) and bool(event.get('action'))

    return {
        'event_id': event['event_id'],
        'is_valid': is_valid,
        'reason': 'ok' if is_valid else 'missing required fields',
    }


def validator_process(connection: mp.connection.Connection) -> None:
    while True:
        event = connection.recv()

        if event == STOP:
            connection.send({'status': 'validator stopped'})
            break

        result = validate_event(event)
        connection.send(result)

    connection.close()

In [14]:
parent_conn, child_conn = MP.Pipe()

process = MP.Process(target=validator_process, args=(child_conn,))
process.start()

events = [
    {'event_id': '1', 'user_id': 'user-1', 'action': 'login'},
    {'event_id': '2', 'user_id': '', 'action': 'payment'},
    {'event_id': '3', 'user_id': 'user-3', 'action': ''},
    {'event_id': '4', 'user_id': 'user-4', 'action': 'logout'},
]

for event in events:
    parent_conn.send(event)
    result = parent_conn.recv()
    print(f'Event {result["event_id"]}: {result["is_valid"]} ({result["reason"]})')

parent_conn.send(STOP)
print(parent_conn.recv())

process.join()
parent_conn.close()

Event 1: True (ok)
Event 2: False (missing required fields)
Event 3: False (missing required fields)
Event 4: True (ok)
{'status': 'validator stopped'}


Тут уже є повноцінний обмін у два боки: запит і відповідь.
Це зручно там, де один окремий процес виконує роль спеціалізованого обробника.

## Спільний стан: `Value`, `Array`, `Lock`

Іноді процесам треба не просто перекинути повідомлення, а змінювати спільні дані.
У такому випадку важливо контролювати одночасний доступ.

In [15]:
from multiprocessing.sharedctypes import Synchronized, SynchronizedArray
from multiprocessing.synchronize import Lock

In [16]:
def modify_shared_memory(shared_value: Synchronized, shared_array: SynchronizedArray, lock: Lock):
    for i in range(len(shared_array)):
        with lock:
            shared_value.value += 1
            shared_array[i] += 1
            print(
                f'Процес {mp.current_process().name}: '
                f'shared_value={shared_value.value}, '
                f'shared_array={list(shared_array)}'
            )
        time.sleep(0.5)

In [17]:
lock = MP.Lock()
shared_value = MP.Value('i', 0)
shared_array = MP.Array('i', [1, 2, 3, 4, 5])

processes = [
    MP.Process(
        target=modify_shared_memory,
        args=(shared_value, shared_array, lock),
    )
    for _ in range(3)
]

for process in processes:
    process.start()

for process in processes:
    process.join()

print(f'Фінальне значення shared_value: {shared_value.value}')
print(f'Фінальне значення shared_array: {list(shared_array)}')

Процес ForkProcess-20: shared_value=1, shared_array=[2, 2, 3, 4, 5]
Процес ForkProcess-21: shared_value=2, shared_array=[3, 2, 3, 4, 5]
Процес ForkProcess-22: shared_value=3, shared_array=[4, 2, 3, 4, 5]
Процес ForkProcess-20: shared_value=4, shared_array=[4, 3, 3, 4, 5]
Процес ForkProcess-21: shared_value=5, shared_array=[4, 4, 3, 4, 5]
Процес ForkProcess-22: shared_value=6, shared_array=[4, 5, 3, 4, 5]
Процес ForkProcess-20: shared_value=7, shared_array=[4, 5, 4, 4, 5]
Процес ForkProcess-21: shared_value=8, shared_array=[4, 5, 5, 4, 5]
Процес ForkProcess-22: shared_value=9, shared_array=[4, 5, 6, 4, 5]
Процес ForkProcess-20: shared_value=10, shared_array=[4, 5, 6, 5, 5]
Процес ForkProcess-21: shared_value=11, shared_array=[4, 5, 6, 6, 5]
Процес ForkProcess-22: shared_value=12, shared_array=[4, 5, 6, 7, 5]
Процес ForkProcess-20: shared_value=13, shared_array=[4, 5, 6, 7, 6]
Процес ForkProcess-21: shared_value=14, shared_array=[4, 5, 6, 7, 7]
Процес ForkProcess-22: shared_value=15, sha

`Lock` тут захищає дані від конфліктів під час запису.
Без нього кілька процесів могли б одночасно змінювати спільні значення і псувати результат.

## `SharedMemory` - спільна памʼять

Цей інструмент потрібен тоді, коли процеси мають працювати з одним і тим самим буфером даних.
Це особливо корисно для великих об'єктів, які не хочеться щоразу копіювати.

### Простий текстовий приклад

Почнемо з малого: один процес записує байти в shared memory, інший їх читає.

In [18]:
def shared_text_worker(shm_name: str, size: int) -> None:
    shm = SharedMemory(name=shm_name)

    try:
        payload = bytes(shm.buf[:size]).decode()
        print('Child reads:', payload)
    finally:
        shm.close()

In [19]:
payload = b'Hello World'
shm = SharedMemory(create=True, size=len(payload))
shm.buf[: len(payload)] = payload

process = MP.Process(target=shared_text_worker, args=(shm.name, len(payload)))

process.start()
process.join()

shm.close()
shm.unlink()

Child reads: Hello World


Це найпростіший спосіб побачити суть shared memory.
Дані не пересилаються повідомленням, а лежать у спільному буфері, доступному різним процесам.

### Shared memory для умовного зображення

Тепер приклад ближчий до практики: один процес заповнює буфер пікселями, інший аналізує яскравість.

In [20]:
WIDTH = 1_000
HEIGHT = 1_000
SIZE = WIDTH * HEIGHT

In [21]:
def fill_image(shared_memory_name: str, size: int) -> None:
    shared_memory = SharedMemory(name=shared_memory_name)

    try:
        buffer = shared_memory.buf

        for index in range(size):
            buffer[index] = random.randint(0, 255)

        print('Producer: image was generated')
    finally:
        shared_memory.close()


def analyze_image(shared_memory_name: str, size: int) -> None:
    shared_memory = SharedMemory(name=shared_memory_name)

    try:
        buffer = shared_memory.buf

        total_brightness = 0

        for index in range(size):
            total_brightness += buffer[index]

        average_brightness = total_brightness / size
        status = 'dark' if average_brightness < 80 else 'normal'

        print(f'Analyzer: average brightness = {average_brightness:.2f}')
        print(f'Analyzer: image status = {status}')
    finally:
        shared_memory.close()

In [22]:
shared_memory = SharedMemory(create=True, size=SIZE)

try:
    producer = MP.Process(target=fill_image, args=(shared_memory.name, SIZE))
    analyzer = MP.Process(target=analyze_image, args=(shared_memory.name, SIZE))

    producer.start()
    producer.join()

    analyzer.start()
    analyzer.join()
finally:
    shared_memory.close()
    shared_memory.unlink()

Producer: image was generated
Analyzer: average brightness = 127.54
Analyzer: image status = normal


Сенс цього прикладу в тому, що великий масив даних не треба щоразу передавати між процесами.
Обидва процеси працюють з однією і тією ж ділянкою пам'яті.

## Сокети: окремі процеси можуть взаємодіяти і через мережу

Це приклад із TCP-сервером і клієнтом.
Його зручно читати як два окремі фрагменти: один запускає сервер, інший підключається до нього.

In [23]:
class Server:
    def __init__(self, host: str, port: int):
        self.host = host
        self.port = port

    def start(self):
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.bind((self.host, self.port))

        sock.listen()
        print(f'Сервер розпочав роботу на порту {self.port}')

        while True:
            client, address = sock.accept()
            print(f'Встановлено зʼєднання з клієнтом {client} на порту {address}')
            message = 'Привід від сервера!!'
            client.send(message.encode())
            client.close()

In [24]:
class Client:
    BUFFER_SIZE = 1024

    def __init__(self, host: str, port: int):
        self.host = host
        self.port = port

    def connect(self):
        client_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        client_sock.connect((self.host, self.port))

        message = client_sock.recv(self.BUFFER_SIZE)
        client_sock.close()
        print(message.decode())

Щоб перевірити цей приклад, сервер і клієнт краще запускати окремо.
Спочатку запускають `Server('127.0.0.1', 9999).start()`, а потім в іншому процесі або терміналі викликають `Client('127.0.0.1', 9999).connect()`.

## `Manager`: спільні Python-структури

`Manager` корисний тоді, коли треба ділити між процесами список, словник або іншу більш "звичну" структуру.
Це зручно, хоча зазвичай не так швидко, як нижчорівневі механізми.

### Спільний список

Чому тут використано окремий файл з worker-функцією.

Коли процес стартує через `spawn`, Python заново імпортує модуль, з якого бере цільову функцію. Якщо код створення процесів лежить просто на верхньому рівні без `if __name__ == "__main__":`, дочірній процес може ще раз виконати весь цей код. Це інколи призводить до рекурсивного створення процесів або до помилок запуску.

У Jupyter це особливо помітно, бо code cells не є звичайними `.py`-модулями. Тому для прикладів з `multiprocessing.Process`, `Manager` або `Pool` часто надійніше винести worker-функцію в окремий `.py`-файл або запускати такі приклади як звичайні скрипти.

Щоб юпітер все-таки запрацював, і можна було б це запустити, додаю це саме так.

In [25]:
from multiprocessing import Manager, Process

In [26]:
from pathlib import Path

worker_file = Path('manager_workers.py')
worker_file.write_text(
    """
def manager_list_worker(shared_list, value: int) -> None:
    shared_list.append(value)
    print(f"Процес додав значення {value}")
""",
    encoding='utf-8',
)

133

In [27]:
from manager_workers import manager_list_worker

manager = Manager()
shared_list = manager.list()

processes = []

for i in range(5):
    process = Process(target=manager_list_worker, args=(shared_list, i))
    processes.append(process)
    process.start()

for process in processes:
    process.join()

print(f'Результат у спільному списку: {list(shared_list)}')

Процес додав значення 0
Процес додав значення 1
Процес додав значення 2
Процес додав значення 4
Процес додав значення 3
Результат у спільному списку: [0, 1, 2, 4, 3]


Тут процеси працюють зі спільним списком майже так, ніби це звичайний Python-об'єкт.
Це зручно для простих сценаріїв, де важливіше зрозумілість, ніж максимальна швидкість.

### Спільний словник зі статистикою

In [28]:
def process_order(order: dict[str, int | str], stats, lock) -> None:
    status = order['status']
    amount = int(order['amount'])

    with lock:
        stats['processed'] += 1

        if status == 'paid':
            stats['paid'] += 1
            stats['revenue'] += amount
        else:
            stats['failed'] += 1

In [29]:
orders = [
    {'id': 1, 'status': 'paid', 'amount': 120},
    {'id': 2, 'status': 'failed', 'amount': 80},
    {'id': 3, 'status': 'paid', 'amount': 250},
    {'id': 4, 'status': 'paid', 'amount': 60},
    {'id': 5, 'status': 'failed', 'amount': 40},
    {'id': 6, 'status': 'paid', 'amount': 90},
]

with MP.Manager() as manager:
    stats = manager.dict(
        {
            'processed': 0,
            'paid': 0,
            'failed': 0,
            'revenue': 0,
        }
    )
    lock = manager.Lock()

    processes = [MP.Process(target=process_order, args=(order, stats, lock)) for order in orders]

    for process in processes:
        process.start()

    for process in processes:
        process.join()

    print(dict(stats))

{'processed': 6, 'paid': 4, 'failed': 2, 'revenue': 520}


Це типовий сценарій агрегації: багато процесів обробляють окремі об'єкти, а потім оновлюють спільний підсумок.
Тут важливо, що доступ до статистики захищається блокуванням.

## Підсумок

Після цих прикладів корисно запам'ятати таку схему:

- `Pool` і `ProcessPoolExecutor` добре підходять для CPU-bound задач;
- `Queue` зручна для розподілу задач між воркерами;
- `Pipe` підходить для прямого каналу між двома процесами;
- `Value`, `Array` і `Lock` потрібні для простого спільного стану;
- `SharedMemory` корисна для великих буферів без зайвого копіювання;
- `Manager` дає змогу ділити між процесами більш звичні Python-структури.